# Week 05 | From sample to reads: designing an experiment

**Core practical: 45 minutes.** D1: compare affordable RNA-seq designs without confusing depth with replication.

No paid AI tool, local installation or external dataset download is required. In Colab, upload this notebook through File > Upload notebook, then run cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

**Project milestone:** D1 submission: completed notebook, <=1200-word design memo, one design table, GENOMES audit and AI/no-AI disclosure. Due end of Week 6; 12% of course. See project brief for rubric.

## Before running (5 min)
DNA sequencing measures sequence fragments; RNA sequencing samples molecules derived from RNA.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Read the D1 brief before editing any parameter.
2. Compute cost and estimated usable fragments for three designs.
3. Compare independent cultures, balance, confounding and read depth.
4. Use the separate coverage example only to check units, not to set an RNA-seq threshold.

In [ ]:
# ALL prices and yields below are hypothetical teaching assumptions, not vendor quotes.
BUDGET = 6000
LIBRARY_COST = 150
COST_PER_MILLION_PAIRS = 10
USABLE_FRACTION = 0.8
DESIGNS = [
    {"name": "A", "n_per_condition": 2, "million_pairs_per_sample": 80},
    {"name": "B", "n_per_condition": 4, "million_pairs_per_sample": 30},
    {"name": "C", "n_per_condition": 6, "million_pairs_per_sample": 15},
]
for d in DESIGNS:
    d["samples"] = 2 * d["n_per_condition"]
    d["cost"] = d["samples"] * (LIBRARY_COST + COST_PER_MILLION_PAIRS*d["million_pairs_per_sample"])
    d["usable_million_pairs"] = USABLE_FRACTION*d["million_pairs_per_sample"]
    d["within_budget"] = d["cost"] <= BUDGET
    print(d)
# Separate toy genome example: one million paired fragments, 2 x 150 bases, genome 5 Mb.
nominal_depth = 1_000_000 * 2 * 150 / 5_000_000
usable_depth = nominal_depth * USABLE_FRACTION
print("Toy DNA genome coverage:", nominal_depth, "nominal;", usable_depth, "usable estimate")
optional_plot([d["name"] for d in DESIGNS], [d["usable_million_pairs"] for d in DESIGNS],
              "Estimated usable million read pairs / sample", "D1 hypothetical design options")
RESULTS = {"data_status": "SYNTHETIC scenario; hypothetical cost model", "designs": DESIGNS,
           "nominal_toy_DNA_depth": nominal_depth, "usable_toy_DNA_depth": usable_depth}

## Explain the evidence (10 min)
**Q1.** Recommend A, B, C or a justified alternative and show the budget calculation.

**Q2.** Specify biological replicates, controls, allocation to batches and one exclusion rule fixed before inspecting outcomes.

**Q3.** Explain why the toy DNA mean-depth calculation neither guarantees per-base coverage nor proves sufficient RNA-seq power.

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, restart the runtime and run all. Download the `.ipynb` and generated summary JSON. In Colab the JSON is in the Files sidebar. Upload both to the course LMS assignment. Do not email patient data. A completion flag checks presence of responses, not scientific correctness. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":5, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W05_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

## Paper / device-free route
Calculate A:4x(150+800)=3800; B:8x(150+300)=3600; C:12x(150+150)=3600. Usable pairs/sample:64,24,12 million. Write a balanced batch-allocation table.

## Optional extension
Optional: derive the ideal Poisson zero-coverage probability exp(-C), then explain why real coverage may violate that model.

## Sources
- [S03] SAM/BAM, VCF and BED specifications, maintained by the HTS community. https://samtools.github.io/hts-specs/
- [S10] DESeq2: Analyzing RNA-seq data with DESeq2, release vignette. https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html